# 📖 Notebook 2: Cross-Region Replication

**Goal**: Understand how data is replicated between Azure paired regions for disaster recovery, and how failover works — all while keeping data within the EU.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why replication between paired regions is necessary
- The difference between synchronous and asynchronous replication
- How Azure handles failover between paired regions
- How to implement async replication that respects data residency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/gdpr-paired-regions
docker compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Open two tabs — one for EU-West (`postgres-eu-west:5432`, DB `gdpr_eu_west`) and one for EU-North (`postgres-eu-north:5432`, DB `gdpr_eu_north`)

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
from datetime import datetime

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 55434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Test connections
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

## 1. Why Replicate Between Paired Regions?

Imagine your primary database is in **West Europe (Netherlands)**. What happens if:

- 🌊 The data center floods?
- ⚡ A power grid fails?
- 🔥 A fire destroys hardware?

Without replication, **all your data is gone**. With paired region replication:

```
Normal Operation:
  Users ──► EU-West (Primary) ──async──► EU-North (Replica)
                    │                          │
               Reads + Writes              Read-only backup

After Disaster in EU-West:
  Users ──────────────────────────────► EU-North (New Primary)
                                              │
                                     Now accepts writes
```

### Key Point for GDPR

Both EU-West and EU-North are in the EU, so a failover between them is not a
transfer to a third country and does not need a Chapter V mechanism.

Two corrections to the folk version of this, though:

- **"Paired regions are always in the same geography" is false.** Microsoft says
  *almost all* regions share a geography with their pair. Brazil South is paired
  with South Central US. If your architecture leans on the pair for a residency
  property, you have to check *your* pair, not the general claim.
- **Being in the EU is not the whole residency question.** Who can *access* the
  replica matters too: a support engineer outside the EU with a console into the
  Irish database is a transfer even though the bytes never moved. This lab does
  not model access control at all.

## 2. Sync vs Async Replication

| Type | How It Works | Speed | Safety | Use Case |
|------|-------------|-------|--------|----------|
| **Synchronous** | Write waits for BOTH regions to confirm | Slow (100-200ms extra) | Zero data loss | Financial transactions |
| **Asynchronous** | Write confirms immediately, replica catches up later | Fast (no extra latency) | Small data loss possible | Most applications |

Cross-region replication in Azure is generally **asynchronous**, because:
- Amsterdam → Dublin is roughly **750 km** great-circle. Light in fibre covers ~200 km/ms, and real paths are not straight, so a round trip between West Europe and North Europe lands in the ballpark of **15–25 ms**.
- Synchronous replication would add a full round trip to *every* commit.
- For most workloads, losing the last few seconds of writes in a genuine regional disaster is a better trade than paying that latency on every write forever.

The price of that trade is a **data-loss window**. Most tutorials mention it and move on. We are going to make it actually lose a record, further down.

Let's build the replication first:

In [ ]:
# ── Simulating Asynchronous Replication ────────────────────
# In Azure, this is handled by the database service itself.
# Here we simulate it with application-level replication.

def replicate_user_async(user_id, source_region, target_region):
    """
    Simulates async replication from one region to another.
    
    In Azure, this is handled by:
    - Azure SQL Geo-Replication (for Azure SQL)
    - Azure Database for PostgreSQL Read Replicas
    - Azure Cosmos DB Multi-Region Writes
    
    We simulate it at the application level.
    """
    source_conn = get_connection(source_region)
    target_conn = get_connection(target_region)

    try:
        # Read from source
        src_cur = source_conn.cursor()
        src_cur.execute("""
            SELECT email, full_name, phone, date_of_birth, country_code,
                   home_region, consent_given, consent_date
            FROM users WHERE id = %s
        """, (user_id,))
        user_data = src_cur.fetchone()

        if not user_data:
            print(f"❌ User {user_id} not found in {source_region}")
            return

        # Simulate network latency (Netherlands → Ireland ≈ 10ms)
        print(f"   📡 Replicating across network ({source_region} → {target_region})...")
        time.sleep(0.01)  # 10ms simulated latency

        # Write to target (using UPSERT to handle re-runs).
        #
        # Every column we selected must appear in the DO UPDATE list. An earlier
        # version of this cell only carried full_name and phone forward, which
        # meant a consent WITHDRAWAL (consent_given TRUE -> FALSE) replicated as
        # a no-op: the primary would show consent revoked while the replica
        # still said TRUE. Under Article 7(3) withdrawal must be as easy as
        # giving consent — a replica that quietly keeps the old value is a real
        # bug, not a cosmetic one, and it is exactly the kind of partial-column
        # replication that survives code review because the demo still "works".
        tgt_cur = target_conn.cursor()
        tgt_cur.execute("""
            INSERT INTO users (email, full_name, phone, date_of_birth, country_code,
                               home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (email) DO UPDATE SET
                full_name     = EXCLUDED.full_name,
                phone         = EXCLUDED.phone,
                date_of_birth = EXCLUDED.date_of_birth,
                country_code  = EXCLUDED.country_code,
                home_region   = EXCLUDED.home_region,
                consent_given = EXCLUDED.consent_given,
                consent_date  = EXCLUDED.consent_date,
                updated_at    = NOW()
            RETURNING id
        """, user_data)
        replica_id = tgt_cur.fetchone()[0]

        # Log the replication event in BOTH regions
        for conn, region in [(source_conn, source_region), (target_conn, target_region)]:
            cur = conn.cursor()
            cur.execute("""
                INSERT INTO data_residency_log
                    (user_id, action, source_region, target_region, table_name, record_id, reason)
                VALUES (%s, 'replicate', %s, %s, 'users', %s,
                        'Async replication for disaster recovery — within EU geography')
            """, (user_id, source_region, target_region, user_id))
            conn.commit()

        print(f"   ✅ User '{user_data[1]}' replicated to {target_region} (replica row id: {replica_id})")
        if replica_id != user_id:
            print(f"      ⚠️  note: source id={user_id}, replica id={replica_id} — the SERIAL")
            print(f"          sequences in the two databases are independent. Never key a")
            print(f"          cross-region operation on a local autoincrement id.")
        return replica_id

    finally:
        source_conn.close()
        target_conn.close()


# Demo: Replicate EU-West users to EU-North
print("🔄 Async Replication: EU-West → EU-North")
print("=" * 50)
print("(Simulates Azure SQL Geo-Replication)\n")

# Get some EU-West users to replicate
conn = get_connection("eu-west")
cur = conn.cursor()
# ORDER BY id so re-runs pick the same three users — an unordered LIMIT is a
# coin flip, and a lab that replicates different rows each run is not a lab.
cur.execute("SELECT id, full_name FROM users WHERE home_region = 'eu-west' ORDER BY id LIMIT 3")
users_to_replicate = cur.fetchall()
conn.close()

for user_id, name in users_to_replicate:
    print(f"\n👤 Replicating user {user_id} ({name}):")
    replicate_user_async(user_id, "eu-west", "eu-north")


# Verify replication actually landed, and landed *completely* — comparing every
# replicated column, not just "a row with that email exists".
REPLICATED_COLS = ("email", "full_name", "phone", "date_of_birth",
                   "country_code", "home_region", "consent_given", "consent_date")

def fetch_replicated(email, region):
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute(f"SELECT {', '.join(REPLICATED_COLS)} FROM users WHERE email = %s", (email,))
    row = cur.fetchone()
    conn.close()
    return row

conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("SELECT email FROM users WHERE id = ANY(%s)", ([u[0] for u in users_to_replicate],))
replicated_emails = [r[0] for r in cur.fetchall()]
conn.close()

for email in replicated_emails:
    west = fetch_replicated(email, "eu-west")
    north = fetch_replicated(email, "eu-north")
    assert north is not None, f"{email} never reached eu-north"
    mismatches = [c for c, w, n in zip(REPLICATED_COLS, west, north) if w != n]
    assert not mismatches, (
        f"replication of {email} was INCOMPLETE — these columns differ between "
        f"regions: {mismatches}. Partial replication of consent or country data "
        f"is a correctness bug, not a rounding error."
    )
print(f"\n✅ All {len(replicated_emails)} replicated users match on every replicated column.")

## 3. Replication Lag — The Tradeoff

With async replication, there's always a **replication lag** — a window of time where the replica is behind the primary.

```
Timeline:
  t=0ms    User writes to EU-West       ✅ Primary has data
  t=0ms    Write confirmed to user       ⏳ Replica doesn't have it yet
  t=20ms   Data arrives at EU-North      ✅ Replica has data

  If EU-West is destroyed between t=0ms and t=20ms, that write is GONE —
  even though the user saw "success".
```

**Real Azure numbers, per service** (they are not one global figure, and the
lab's earlier "< 5 seconds everywhere" was made up):

| Mechanism | Documented figure |
|---|---|
| Azure SQL DB, failover groups / active geo-replication | RTO *typically less than 60 seconds*; RPO *"equal to or greater than 0 — depends on data changes that haven't been replicated"* |
| Azure SQL DB, geo-restore from geo-redundant backups | RTO and RPO *typically minutes or hours* |
| Geo-redundant storage (GRS) | Typically under 15 minutes, **with no SLA on replication time**. The opt-in *geo priority replication* feature is what turns that into a 15-minute commitment for block blobs. |

Note that Azure SQL's RPO is expressed as "≥ 0", not as a number. That is the
honest way to state an asynchronous RPO: the window is however far behind the
replica happens to be when the primary dies.

Let's time our own simulated replication — carefully:

In [ ]:
# ── Measuring Replication Lag ──────────────────────────────

def measure_replication_lag():
    """
    Writes a record to EU-West and measures how long until
    it appears in EU-North.
    """
    # Write to EU-West
    west_conn = get_connection("eu-west")
    west_cur = west_conn.cursor()

    timestamp = datetime.now().isoformat()
    test_email = f"lag-test-{timestamp}@example.com"

    write_start = time.time()
    west_cur.execute("""
        INSERT INTO users (email, full_name, phone, country_code, home_region, consent_given)
        VALUES (%s, 'Lag Test User', '+1-555-0000', 'NL', 'eu-west', TRUE)
        RETURNING id
    """, (test_email,))
    user_id = west_cur.fetchone()[0]
    west_conn.commit()
    write_time = (time.time() - write_start) * 1000

    print(f"✏️  Write to EU-West: {write_time:.1f}ms (user_id={user_id})")

    # Now replicate (this is what Azure does automatically)
    repl_start = time.time()
    replicate_user_async(user_id, "eu-west", "eu-north")
    repl_time = (time.time() - repl_start) * 1000

    print(f"⏱️  Simulated end-to-end replication: {repl_time:.1f}ms")
    print( "   ⚠️  Do NOT read this as an Azure replication lag figure. It is dominated")
    print( "       by two fresh psycopg2 connections plus six SQL round trips against a")
    print( "       laptop, and it contains a hard-coded 10ms sleep. Real geo-replication")
    print( "       happens inside the storage engine on an already-open channel.")
    print( "       The only thing that transfers to reality is the SHAPE: the write is")
    print( "       acknowledged first, and the replica catches up afterwards.")
    print(f"   Total time until the row exists in both regions: {write_time + repl_time:.1f}ms")

    # The window must be non-zero, or the next demo has nothing to lose.
    assert repl_time > 0, "replication took no measurable time — the lag window is the lesson"

    # Clean up test user
    west_conn.close()
    for region in ["eu-west", "eu-north"]:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("DELETE FROM users WHERE email = %s", (test_email,))
        conn.commit()
        conn.close()

print("📊 Replication Lag Test")
print("=" * 50)
measure_replication_lag()

## 3b. Actually losing a write (the part tutorials skip)

Every article about asynchronous replication contains a sentence like *"there is
a small window in which data can be lost."* Almost none of them show the loss.
That is a problem, because a reader who has never seen the record disappear does
not really believe it, and does not design for it.

So: we will write two users to EU-West. One gets replicated. The other is still
sitting in the replication window when the region dies. Then we fail over and
count who survived.

The number of survivors is your **RPO, measured in records instead of seconds** —
which is the unit your users actually experience.


In [ ]:
# ── Demonstrating the RPO window: a write that does NOT survive ────────

RPO_SAFE = "rpo.replicated@example.nl"      # written, then replicated
RPO_LOST = "rpo.in-flight@example.nl"       # written, disaster strikes first

def _purge(emails):
    for region in ["eu-west", "eu-north"]:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("DELETE FROM users WHERE email = ANY(%s)", (list(emails),))
        conn.commit()
        conn.close()

_purge([RPO_SAFE, RPO_LOST])

def write_to_primary(email, name):
    """A normal user-facing write. Commits locally and returns success."""
    conn = get_connection("eu-west")
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO users (email, full_name, country_code, home_region, consent_given, consent_date)
        VALUES (%s, %s, 'NL', 'eu-west', TRUE, NOW())
        RETURNING id
    """, (email, name))
    uid = cur.fetchone()[0]
    conn.commit()
    conn.close()
    print(f"   ✅ 200 OK returned to the user — '{name}' committed in eu-west (id={uid})")
    return uid

print("⏱️  RPO WINDOW DEMONSTRATION")
print("=" * 60)

print("\n1️⃣  A write that completes its replication:")
safe_id = write_to_primary(RPO_SAFE, "Replicated Renske")
replicate_user_async(safe_id, "eu-west", "eu-north")

print("\n2️⃣  A write that is still in the replication window:")
lost_id = write_to_primary(RPO_LOST, "In-Flight Ingeborg")
print("   📡 replication scheduled...")
print("   🔥 ...and EU-West is destroyed before it runs.")
# NOTE: we simply never call replicate_user_async() for this row. That is
# exactly what an unreplicated write is — nothing dramatic happens, the work
# just never gets done, and nobody finds out until the failover.

print("\n3️⃣  Failover. EU-North is now the only surviving copy.")
print("    What does it have?")

conn = get_connection("eu-north")
cur = conn.cursor()
cur.execute("SELECT email FROM users WHERE email = ANY(%s)", ([RPO_SAFE, RPO_LOST],))
survivors = {r[0] for r in cur.fetchall()}
conn.close()

for email in (RPO_SAFE, RPO_LOST):
    mark = "✅ survived" if email in survivors else "❌ LOST"
    print(f"    {mark:<12} {email}")

# ── This is the lesson, so make the notebook fail loudly if it stops holding ──
assert RPO_SAFE in survivors, (
    "the replicated write should have survived the failover — if it didn't, "
    "replication itself is broken and this demo is measuring the wrong thing"
)
assert RPO_LOST not in survivors, (
    "the in-flight write SURVIVED the failover, which means this notebook is no "
    "longer demonstrating the asynchronous data-loss window at all"
)

print(f"\n📉 RPO for this incident: 1 record.")
print( "    The user who created 'In-Flight Ingeborg' saw a 200 OK. Their account")
print( "    no longer exists. No error was raised anywhere; nothing retried.")

print("\n💡 What this means for the rest of the lab:")
print("   • Under GDPR this cuts both ways. A lost *write* is a durability problem.")
print("     A lost *delete* is a compliance problem: if an erasure is executed on")
print("     the primary and the region dies before it replicates, the failover")
print("     promotes a replica that still holds the person's data — and your")
print("     erasure_requests table cheerfully says 'completed'.")
print("   • That is exactly the scenario Notebook 3 has to handle, and exactly")
print("     what Notebook 4's audit has to be able to catch.")
print("   • Synchronous replication removes this window and costs you a round trip")
print("     on every commit. That is the whole trade. There is no third option.")

_purge([RPO_SAFE, RPO_LOST])
print("\n🧹 RPO demo data cleaned up.")


## 4. Failover Simulation

When the primary region goes down, the system must **failover** to the paired region. Here's how Azure handles it:

1. Azure detects the primary is unhealthy
2. DNS is updated to point to the secondary region
3. The secondary is promoted to primary (accepts writes)
4. When the original region recovers, it becomes the new secondary

Let's simulate this:

In [ ]:
# ── Failover Simulation ────────────────────────────────────

class PairedRegionManager:
    """
    Simulates Azure's paired region failover mechanism.
    
    In Azure, this is handled by:
    - Azure Traffic Manager (DNS-based failover)
    - Azure SQL Auto-Failover Groups
    - Azure Front Door (HTTP-based routing)
    """

    def __init__(self):
        self.primary = "eu-west"
        self.secondary = "eu-north"
        # Which regions we have *simulated* as unreachable. We cannot actually
        # stop a container from a notebook (other labs share this host), so an
        # "outage" here means: refuse to talk to that region, and make
        # health_check() report it as down. The important part is that
        # get_active_region() derives the answer from a probe rather than from
        # a flag someone remembered to flip.
        self._simulated_outage = set()

    def health_check(self, region):
        """Probes a region. Returns False if unreachable (or simulated down)."""
        if region in self._simulated_outage:
            return False
        try:
            conn = get_connection(region)
            cur = conn.cursor()
            cur.execute("SELECT 1")
            cur.fetchone()
            conn.close()
            return True
        except Exception:
            return False

    def get_active_region(self):
        """
        Returns the region that should handle writes, based on an actual health
        probe. An earlier version of this class returned a cached boolean that
        nothing ever set from a probe, and defined health_check() without ever
        calling it — so the 'failover' was a variable assignment, and a genuinely
        dead primary would not have been noticed at all.
        """
        if self.health_check(self.primary):
            return self.primary
        if self.health_check(self.secondary):
            return self.secondary
        raise RuntimeError("both regions are unreachable — this is a real outage")

    def simulate_failure(self):
        """Simulates EU-West going down."""
        print("\n🔥 DISASTER: EU-West region is DOWN!")
        print("   (In reality: data center flood, power outage, etc.)")
        self._simulated_outage.add(self.primary)
        assert not self.health_check(self.primary), "primary should now probe unhealthy"
        print(f"   Health probe on {self.primary}: DOWN")
        print(f"   Active region is now: {self.get_active_region()}")

    def simulate_recovery(self):
        """Simulates EU-West coming back online."""
        print("\n✅ RECOVERY: EU-West region is back online!")
        self._simulated_outage.discard(self.primary)
        assert self.health_check(self.primary), "primary should probe healthy again"
        print(f"   Health probe on {self.primary}: UP")
        print(f"   Active region is now: {self.get_active_region()}")

    def write_user(self, email, full_name, country_code, home_region):
        """
        Writes a user to whichever region is currently active.

        `home_region` is the user's JURISDICTION assignment and is passed in by
        the caller (from the geo-router). It is deliberately NOT set to the
        active region: failing over must not silently relabel a user's home
        region. An earlier version stored home_region = active, which meant a
        German customer written during an outage was permanently recorded as
        belonging to eu-north — corrupting the very field Notebook 4 audits.
        """
        active = self.get_active_region()
        if active != home_region:
            print(f"   ⚠️  writing a {home_region} user into {active} (failover) —")
            print(f"       the row is displaced, the label is not.")
        conn = get_connection(active)
        cur = conn.cursor()

        cur.execute("""
            INSERT INTO users (email, full_name, country_code, home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, TRUE, NOW())
            ON CONFLICT (email) DO NOTHING
            RETURNING id
        """, (email, full_name, country_code, home_region))

        result = cur.fetchone()
        conn.commit()
        conn.close()

        if result:
            print(f"   ✏️  Wrote '{full_name}' to {active} (ID: {result[0]})")
        else:
            print(f"   ⏭️  '{full_name}' already exists in {active}")
        return active


# ── Run the failover scenario ──────────────────────────────

# Start from a clean slate so re-running this cell is idempotent — otherwise a
# partial re-run leaves rows from the previous attempt in the wrong region and
# the placement assertions below fail for the wrong reason.
for _region in ["eu-west", "eu-north"]:
    _c = get_connection(_region)
    _cur = _c.cursor()
    _cur.execute("DELETE FROM users WHERE email LIKE 'failover.test%'")
    _c.commit(); _c.close()

manager = PairedRegionManager()

print("🏗️  Paired Region Failover Simulation")
print("=" * 50)

# Phase 1: Normal operation
print("\n📍 Phase 1: Normal Operation")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test1@example.de", "Test User 1", "DE", home_region="eu-west")

# Phase 2: EU-West goes down
manager.simulate_failure()
print(f"\n📍 Phase 2: Failover Active")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test2@example.de", "Test User 2", "DE", home_region="eu-west")

# Phase 3: EU-West recovers
manager.simulate_recovery()
print(f"\n📍 Phase 3: Recovery Complete")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test3@example.de", "Test User 3", "DE", home_region="eu-west")

# The failover must actually have moved the write, or there is no demo here.
assert manager.get_active_region() == "eu-west", "should be back on the primary after recovery"

print("\n💡 Key insight: during ALL phases the data stayed inside the EU, because")
print("   both West Europe (NL) and North Europe (IE) are EU regions. A failover")
print("   between them needs no Chapter V transfer mechanism.")
print("\n⚠️  Two things this simulation does NOT show:")
print("   • Failback. Test User 2 was written to eu-north during the outage and is")
print("     still only there. Real systems need a reconciliation pass, and it has")
print("     to handle rows written on BOTH sides during a split brain.")
print("   • A displaced row is still displaced. Test User 2's home_region is")
print("     'eu-west' but the row is physically in eu-north. That is fine here")
print("     (same geography) and would be a reportable transfer if the pair")
print("     crossed a border — as Brazil South ↔ South Central US does.")

In [ ]:
# ── Verify: Where did each write land? ─────────────────────

print("🔍 Verifying Write Locations After Failover")
print("=" * 60)

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT email, full_name, home_region
        FROM users
        WHERE email LIKE 'failover.test%'
        ORDER BY email
    """)
    results = cur.fetchall()
    print(f"\n📦 {region.upper()}:")
    for row in results:
        print(f"   {row[1]} ({row[0]}) — stored in {row[2]}")
    if not results:
        print("   (no failover test users here)")
    conn.close()

# Assert the failover actually displaced exactly the middle write.
placement = {}
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT email, home_region FROM users WHERE email LIKE 'failover.test%'")
    for email, home in cur.fetchall():
        placement[email] = (region, home)
    conn.close()

assert placement["failover.test1@example.de"][0] == "eu-west"
assert placement["failover.test2@example.de"][0] == "eu-north", \
    "the write during the outage should have landed in the secondary"
assert placement["failover.test3@example.de"][0] == "eu-west", \
    "after recovery, writes should return to the primary"
# The label must survive the displacement.
for email, (region, home) in placement.items():
    assert home == "eu-west", (
        f"{email} was relabelled home_region={home} — failover must not rewrite "
        f"a user's jurisdiction assignment just because the row moved"
    )

print("\n💡 Notice:")
print("   - Test User 1 & 3 → EU-West (primary was healthy)")
print("   - Test User 2 → EU-North (written during the outage)")
print("   - ALL THREE still carry home_region='eu-west'. The row moved; the")
print("     jurisdiction did not. Notebook 4 relies on that distinction to tell")
print("     'replicated for DR' apart from 'misrouted'.")
print("   - Nothing has replicated Test User 2 back to EU-West. Failback is real")
print("     work that this lab does not do; we just delete the test rows below.")

## 5. Why Microsoft Uses Paired Region Replication

### What Microsoft actually documents

There is no single "Azure RPO/RTO" number — the figures are per service, and
several of them are *typical values*, not SLAs. Sourced:

| Property | What Microsoft says | Caveat |
|---|---|---|
| **RTO** (Azure SQL DB, failover groups / active geo-replication) | "Typically less than 60 seconds" | Typical, not guaranteed; excludes the time to *decide* to fail over |
| **RPO** (same) | "Equal to or greater than 0 (depends on data changes before the disruptive event that haven't been replicated)" | Deliberately not a number — see the demo above |
| **RTO/RPO** (geo-restore from geo-redundant backups) | "Typically minutes or hours" | Depends on database size |
| **GRS replication** | Typically under 15 minutes, **no SLA on how long geo-replication takes** | The opt-in *geo priority replication* feature adds a 15-minute commitment for block blobs |
| **Data residency** | "To meet data residency requirements, **almost all** regions reside within the same geography as their pair" | Brazil South ↔ South Central US is the documented exception |
| **Update rollouts** | "Azure strives to stagger any planned system updates across region pairs" | *Strives to* — a best-effort operational practice |
| **Failover** | "Deploying resources to a region in a pair doesn't automatically make them more resilient, nor does it provide automatic high availability, disaster recovery capabilities, or failover" | You still design and test your own DR |

### The Sequential Update Practice

Azure staggers planned platform updates across a region pair, so a faulty update
that breaks West Europe should not simultaneously break North Europe. This is a
genuine benefit of pairing and one of the better reasons to use your pair as your
secondary. Note the framing though: Microsoft says it *strives to* stagger
updates. Treat it as a strong operational practice, not a contractual guarantee,
and note that it says nothing about *unplanned* correlated failures — a bad
config push or a shared control-plane dependency does not respect the pairing.

### Cost Implications

Geo-replication costs money — you run two databases, and you pay egress on the
replication traffic. Whether that is worth it is an ordinary availability
decision, and it is worth being precise about *which* risk it addresses:

- **Downtime cost** is the real driver. Published per-hour outage costs vary
  enormously by industry and are mostly vendor-marketing numbers; use your own.
- **GDPR fines are not the argument here.** Article 32 requires "the ability to
  restore the availability and access to personal data in a timely manner in the
  event of a physical or technical incident" — so availability *is* in scope. But
  the €20m/4% headline tier is for things like processing without a lawful basis,
  not for having a single-region database. Reaching for the maximum-fine number
  to justify a DR budget is the kind of argument that gets an architect ignored
  the second time.
- **A second region is not a backup.** Replication faithfully copies your
  `DELETE FROM users` to the replica in milliseconds. You need point-in-time
  backups for that — which is precisely the problem Notebook 3 runs into from
  the other direction.

In [ ]:
# ── Clean up failover test data ────────────────────────────

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email LIKE 'failover.test%'")
    deleted = cur.rowcount
    conn.commit()
    conn.close()
    if deleted > 0:
        print(f"🧹 Cleaned up {deleted} test users from {region}")

print("✅ Test data cleaned up")

## 🎯 Key Takeaways

1. **Paired regions replicate within the same geography — *almost* always.** West Europe ↔ North Europe does; Brazil South ↔ South Central US does not. Check your own pair, and check whether the service you use even uses pairs.
2. **Async replication is the default** because a synchronous commit costs a full cross-region round trip (~15–25 ms Amsterdam↔Dublin) on *every* write.
3. **The lag window loses real records.** We made it lose one: a user got a 200 OK for an account that no longer exists. Azure SQL states its geo-replication RPO as "≥ 0" rather than as a number, for exactly this reason.
4. **Failover must be driven by a probe**, not by a boolean somebody remembers to set. A health-check function that is never called is worse than none, because it looks like the problem is handled.
5. **Failover moves rows; it must not relabel them.** A displaced row keeps its `home_region`. Conflating "where the bytes are" with "whose jurisdiction this is" destroys the only field an audit can use.
6. **Replication is not backup.** It copies your mistakes too, at line rate.
7. **A lost delete is worse than a lost write.** If an erasure is executed on the primary and lost in the replication window, you failover to a replica that still holds the data — while your records say the erasure completed. Notebook 3.

## ⏭️ Next Up

In **Notebook 3**, we'll tackle the **Right to Erasure** (Article 17) — deleting across replicas *and* backups, and untangling the pseudonymisation/anonymisation confusion that makes most "we anonymised it" claims wrong.